# 📊 財務分析ツール
Google Drive のファイルをブラウザから選んで財務指標を自動分析します。

**使い方:** 上から順に ▶️ を押してください。ファイルマネージャーで番号を入力するだけでOKです。

## ① ライブラリをインストール（初回のみ）

In [ ]:
!pip install pymupdf openpyxl python-docx -q
print('✅ インストール完了')

## ② 分析コードを読み込む

In [ ]:
from dataclasses import dataclass
from typing import Optional, Dict, List
import re, os
from pathlib import Path

@dataclass
class IncomeStatement:
    revenue: float
    cost_of_goods_sold: float
    gross_profit: Optional[float] = None
    selling_expenses: float = 0.0
    general_admin_expenses: float = 0.0
    operating_expenses: Optional[float] = None
    operating_income: Optional[float] = None
    non_operating_income: float = 0.0
    non_operating_expenses: float = 0.0
    ordinary_income: Optional[float] = None
    extraordinary_income: float = 0.0
    extraordinary_losses: float = 0.0
    income_before_tax: Optional[float] = None
    income_tax: float = 0.0
    net_income: Optional[float] = None
    period: str = ''
    def __post_init__(self):
        if self.gross_profit is None: self.gross_profit = self.revenue - self.cost_of_goods_sold
        if self.operating_expenses is None: self.operating_expenses = self.selling_expenses + self.general_admin_expenses
        if self.operating_income is None: self.operating_income = self.gross_profit - self.operating_expenses
        if self.ordinary_income is None: self.ordinary_income = self.operating_income + self.non_operating_income - self.non_operating_expenses
        if self.income_before_tax is None: self.income_before_tax = self.ordinary_income + self.extraordinary_income - self.extraordinary_losses
        if self.net_income is None: self.net_income = self.income_before_tax - self.income_tax

KEYWORD_MAP = {
    '売上高':'revenue','売上':'revenue','収益合計':'revenue','営業収益':'revenue',
    '売上原価':'cost_of_goods_sold','原価合計':'cost_of_goods_sold','製造原価':'cost_of_goods_sold',
    '売上総利益':'gross_profit','粗利':'gross_profit','粗利益':'gross_profit',
    '販売費及び一般管理費':'selling_and_admin','販管費':'selling_and_admin',
    '販売費':'selling_expenses','一般管理費':'general_admin_expenses','管理費':'general_admin_expenses',
    '営業利益':'operating_income','営業損益':'operating_income',
    '営業外収益':'non_operating_income',
    '営業外費用':'non_operating_expenses','支払利息':'non_operating_expenses',
    '経常利益':'ordinary_income','経常損益':'ordinary_income',
    '特別利益':'extraordinary_income',
    '特別損失':'extraordinary_losses','特別費用':'extraordinary_losses',
    '税引前当期純利益':'income_before_tax','税引前利益':'income_before_tax',
    '法人税':'income_tax','法人税等':'income_tax',
    '当期純利益':'net_income','当期純損失':'net_income','当期利益':'net_income','純利益':'net_income',
    '営業損失':'operating_income',
    '経常損失':'ordinary_income',
    '税引前当期純損失':'income_before_tax',
    '受取利息':'non_operating_income',
    '法人税、住民税及び事業税':'income_tax',
    '売上収入':'revenue',
}

def _parse_number(text):
    if not text: return None
    t = str(text).strip().replace(',','').replace('，','').replace(' ','')
    if not t or t in ('-','―','—','－'): return 0.0
    neg = False
    if t.startswith(('△','▲','▽')): neg,t = True,t[1:]
    elif t.startswith('(') and t.endswith(')'): neg,t = True,t[1:-1]
    elif t.startswith('（') and t.endswith('）'): neg,t = True,t[1:-1]
    try: v=float(t); return -v if neg else v
    except: return None

def _match_keyword(label):
    label = label.strip()
    if label in KEYWORD_MAP: return KEYWORD_MAP[label]
    for kw in sorted(KEYWORD_MAP.keys(), key=len, reverse=True):
        if kw in label: return KEYWORD_MAP[kw]
    return None

def _build_income(fields, period):
    if 'selling_and_admin' in fields and 'selling_expenses' not in fields:
        c=fields.pop('selling_and_admin'); fields['selling_expenses']=c*0.5; fields['general_admin_expenses']=c*0.5
    else: fields.pop('selling_and_admin',None)
    def g(k): return fields.get(k,0.0)
    kw=dict(period=period,revenue=g('revenue'),cost_of_goods_sold=g('cost_of_goods_sold'),
            selling_expenses=g('selling_expenses'),general_admin_expenses=g('general_admin_expenses'),
            non_operating_income=g('non_operating_income'),non_operating_expenses=g('non_operating_expenses'),
            extraordinary_income=g('extraordinary_income'),extraordinary_losses=g('extraordinary_losses'),income_tax=g('income_tax'))
    for k in ('gross_profit','operating_income','ordinary_income','income_before_tax','net_income'):
        if k in fields: kw[k]=fields[k]
    return IncomeStatement(**kw)

TOTAL_KEYWORDS = ('合計','累計','年計','通期','決算')

def _resolve_total_col(header, month_re):
    for i,h in enumerate(header):
        if any(k in h for k in TOTAL_KEYWORDS): return i
    mc=[i for i,h in enumerate(header) if month_re.search(h)]
    return max(mc) if len(mc)>=3 else None

def load_excel(path, period='', unit=1.0, sheet_name=None):
    import openpyxl
    wb=openpyxl.load_workbook(path,read_only=True,data_only=True)
    ws=wb[wb.sheetnames[0]]
    for name in wb.sheetnames:
        if sheet_name and name==sheet_name: ws=wb[name]; break
        elif not sheet_name and any(k in name for k in ('損益','PL','P&L','売上')): ws=wb[name]; break
    rows=[list(r) for r in ws.iter_rows(values_only=True)]; wb.close()
    if not period:
        for row in rows[:5]:
            for c in row:
                m=re.search(r'(\d{4}年\d{1,2}月期)',str(c or ''))
                if m: period=m.group(1); break
    month_re=re.compile(r'\d{1,2}月')
    header_idx,header=0,[str(c or '').strip() for c in rows[0]]
    for i,row in enumerate(rows[:20]):
        strs=[str(c or '').strip() for c in row]
        if sum(1 for s in strs if month_re.search(s))>=3 or any(k in ' '.join(strs) for k in ('合計','売上高','勘定科目')):
            header_idx,header=i,strs; break
    total_col=_resolve_total_col(header,month_re)
    fields={}
    if total_col is not None:
        for row in rows[header_idx+1:]:
            label=str(row[0] or '').strip(); f=_match_keyword(label)
            if f and total_col<len(row):
                v=_parse_number(row[total_col])
                if v is not None: fields.setdefault(f,v*unit)
    else:
        for row in rows[header_idx+1:]:
            label=str(row[0] or '').strip(); f=_match_keyword(label)
            if f:
                for c in reversed(row[1:]):
                    v=_parse_number(c)
                    if v is not None: fields.setdefault(f,v*unit); break
    return _build_income(fields,period)

def load_pdf(path, period='', unit=1.0):
    import fitz
    all_rows=[]
    with fitz.open(path) as doc:
        for page in doc:
            words=page.get_text('words'); pw=page.rect.width
            lw,nw={},{}
            for x0,y0,x1,y1,text,*_ in words:
                yk=int(y0/6)*6
                (lw if x0<pw*0.4 else nw).setdefault(yk,[]).append(text.strip())
            if not period:
                m=re.search(r'(\d{4}年\d{1,2}月期)',page.get_text())
                if m: period=m.group(1)
            for yk in sorted(lw):
                label=' '.join(lw[yk]); nums=nw.get(yk,[])
                if not nums:
                    for dy in (6,12,-6,-12):
                        nums=nw.get(yk+dy,[])
                        if nums: break
                all_rows.append([label, nums[-1] if nums else ''])
    fields={}
    for row in all_rows:
        f=_match_keyword(row[0])
        if f:
            v=_parse_number(row[1])
            if v is not None: fields.setdefault(f,v*unit)
    return _build_income(fields,period)

def load_word(path, period='', unit=1.0):
    import docx
    doc=docx.Document(path)
    if not period:
        for p in doc.paragraphs[:10]:
            m=re.search(r'(\d{4}年\d{1,2}月期)',p.text)
            if m: period=m.group(1); break
    fields={}; month_re=re.compile(r'\d{1,2}月')
    for table in doc.tables:
        rows=[[c.text.strip() for c in row.cells] for row in table.rows]
        if not rows: continue
        header=rows[0]; total_col=_resolve_total_col(header,month_re)
        _all_fields = set(KEYWORD_MAP.values()) - {'selling_and_admin'}
        for row in rows[1:] if total_col is not None else rows:
            if len(fields) >= len(_all_fields): break
            label=row[0]; f=_match_keyword(label)
            if f:
                cells=[row[total_col]] if total_col is not None and total_col<len(row) else row[1:]
                for c in reversed(cells):
                    v=_parse_number(c)
                    if v is not None: fields.setdefault(f,v*unit); break
    return _build_income(fields,period)

def load_file(path, period='', unit=1.0, sheet_name=None):
    ext=Path(path).suffix.lower()
    if ext=='.pdf': return load_pdf(path,period,unit)
    if ext in ('.xlsx','.xls','.xlsm'): return load_excel(path,period,unit,sheet_name)
    if ext=='.docx': return load_word(path,period,unit)
    raise ValueError(f'非対応形式: {ext}')

def analyze_and_print(is_, company_name=''):
    sep='='*55
    print(sep)
    print(f'  財務分析レポート: {company_name}')
    print(f'  対象期間: {is_.period}')
    print(sep)
    def pct(n,d): return (n/d*100) if d else 0.0
    print('\n【主要数値】')
    print(f'  売上高:       {is_.revenue:>15,.0f} 円')
    print(f'  売上原価:     {is_.cost_of_goods_sold:>15,.0f} 円')
    print(f'  売上総利益:   {is_.gross_profit:>15,.0f} 円')
    print(f'  営業利益:     {is_.operating_income:>15,.0f} 円')
    print(f'  経常利益:     {is_.ordinary_income:>15,.0f} 円')
    print(f'  当期純利益:   {is_.net_income:>15,.0f} 円')
    print('\n【収益性指標】')
    gm=pct(is_.gross_profit,is_.revenue); om=pct(is_.operating_income,is_.revenue); nm=pct(is_.net_income,is_.revenue)
    print(f'  売上総利益率: {gm:6.2f}%  ','★★★' if gm>=50 else '★★' if gm>=30 else '★')
    print(f'  営業利益率:   {om:6.2f}%  ','★★★' if om>=15 else '★★' if om>=5 else '★')
    print(f'  純利益率:     {nm:6.2f}%  ','★★★' if nm>=10 else '★★' if nm>=3 else '★')
    print('\n【評価コメント】')
    if gm>=50: print('  ・粗利率50%超 — 非常に高い価格競争力があります')
    elif gm>=30: print('  ・粗利率は良好な水準です')
    else: print('  ・粗利率が低め。原価管理の見直しを検討してください')
    if om>=15: print('  ・営業利益率15%超 — 優秀な収益力です')
    elif om>=5: print('  ・営業利益率は安定した水準です')
    elif om>=0: print('  ・営業利益率が低め。販管費の削減を検討してください')
    else: print('  ・営業損失が発生しています。早急な改善が必要です')
    print(sep)

print('✅ 分析コード読み込み完了')

## ③ Google Drive に接続
ポップアップが出たら **「Googleドライブに接続」** をタップしてください

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ 接続完了')

## ④ 📁 ファイルマネージャー
ドライブのファイルを一覧表示して番号で選択できます

In [ ]:
import os, re
from pathlib import Path
from IPython.display import display, HTML

# ─── 設定 ───────────────────────────────────
SEARCH_ROOT   = '/content/drive/MyDrive'  # 検索起点（変更可）
SEARCH_DEPTH  = 3                          # フォルダの深さ（1=直下のみ）
SUPPORTED_EXT = {'.pdf', '.xlsx', '.xls', '.xlsm', '.docx'}
# ────────────────────────────────────────────

EXT_ICON = {'.pdf': '📄', '.xlsx': '📊', '.xls': '📊', '.xlsm': '📊', '.docx': '📝'}

def scan_drive(root, max_depth):
    found = []
    def _scan(path, depth):
        if depth > max_depth: return
        try:
            for entry in sorted(os.scandir(path), key=lambda e: (e.is_file(), e.name)):
                if entry.is_dir():
                    _scan(entry.path, depth + 1)
                elif Path(entry.name).suffix.lower() in SUPPORTED_EXT:
                    found.append(entry.path)
        except PermissionError:
            pass
    _scan(root, 1)
    return found

print('📁 ファイルを検索中...')
files = scan_drive(SEARCH_ROOT, SEARCH_DEPTH)

if not files:
    print('⚠️ 対応ファイルが見つかりませんでした。')
    print(f'   検索場所: {SEARCH_ROOT}')
    print(f'   対応形式: {SUPPORTED_EXT}')
else:
    print(f'\n{len(files)} 件のファイルが見つかりました:\n')
    header = f'{'番号':>4}  {'種類':2}  ファイル名（フォルダ）'
    print(header)
    print('-' * 70)
    for i, fpath in enumerate(files, 1):
        p = Path(fpath)
        ext = p.suffix.lower()
        icon = EXT_ICON.get(ext, '📎')
        rel = fpath.replace(SEARCH_ROOT + '/', '')
        display_name = rel if len(rel) <= 60 else '...' + rel[-57:]
        print(f'{i:>4}  {icon}   {display_name}')
    print('-' * 70)
    print('\n👇 次のセルで番号を入力して分析を実行してください')

# グローバルに保持
_fm_files = files

## ⑤ 番号を入力して分析実行
上の一覧で確認した番号を入力してください

In [ ]:
# ==============================
# 番号と設定を入力してください
# ==============================
FILE_NUMBER  = 1           # ← 上の一覧の番号
COMPANY_NAME = '株式会社〇〇'  # ← 企業名
PERIOD       = ''          # ← 会計期間（空欄=自動検出）
UNIT         = 1           # ← 1=円, 1000=千円, 1000000=百万円
SHEET_NAME   = None        # ← Excelシート名（None=自動）
# ==============================

if not globals().get('_fm_files'):
    print('⚠️ 先にセル④を実行してファイル一覧を取得してください')
elif FILE_NUMBER < 1 or FILE_NUMBER > len(_fm_files):
    print(f'⚠️ 番号は 1〜{len(_fm_files)} の範囲で入力してください')
else:
    selected = _fm_files[FILE_NUMBER - 1]
    print(f'選択したファイル: {Path(selected).name}')
    print(f'パス: {selected}\n')
    try:
        is_ = load_file(selected, period=PERIOD, unit=UNIT, sheet_name=SHEET_NAME)
        analyze_and_print(is_, company_name=COMPANY_NAME)
    except Exception as e:
        print(f'❌ エラー: {e}')
        print('\nヒント: セル⑦のデバッグセルを実行してファイル内容を確認してください')

## ⑥ 複数ファイルをまとめて分析

In [ ]:
# ④の一覧番号で複数ファイルを指定
SELECTIONS = [
    {'no': 1, 'company': '会社A', 'unit': 1},
    {'no': 2, 'company': '会社B', 'unit': 1000},
    {'no': 3, 'company': '会社C', 'unit': 1},
]

for sel in SELECTIONS:
    no = sel['no']
    if no < 1 or no > len(_fm_files):
        print(f'⚠️ 番号 {no} は範囲外です'); continue
    fpath = _fm_files[no - 1]
    print(f'\n[{no}] {Path(fpath).name}')
    try:
        is_ = load_file(fpath, unit=sel.get('unit',1))
        analyze_and_print(is_, company_name=sel['company'])
    except Exception as e:
        print(f'  ❌ エラー: {e}')

## ⑦ 🔍 デバッグ（うまく読み込めない場合）

In [ ]:
# ④の番号を入れて実行
DEBUG_FILE_NUMBER = 1

if not globals().get('_fm_files'):
    print('先にセル④を実行してください')
else:
    fpath = _fm_files[DEBUG_FILE_NUMBER - 1]
    ext = Path(fpath).suffix.lower()
    print(f'ファイル: {fpath}\n')

    if ext in ('.xlsx', '.xls', '.xlsm'):
        import openpyxl
        wb = openpyxl.load_workbook(fpath, read_only=True, data_only=True)
        print('シート一覧:', wb.sheetnames)
        ws = wb[wb.sheetnames[0]]
        print('\n先頭15行:')
        for i, row in enumerate(ws.iter_rows(values_only=True)):
            if i >= 15: break
            print(f'  行{i+1:2}: {[str(c)[:15] if c else "" for c in row]}')
        wb.close()

    elif ext == '.pdf':
        import fitz
        with fitz.open(fpath) as doc:
            print('ページ数:', len(doc))
            print('\n1ページ目テキスト（先頭30行）:')
            lines = doc[0].get_text().split('\n')
            for i, line in enumerate(lines[:30]):
                if line.strip(): print(f'  {line}')

    elif ext == '.docx':
        import docx
        doc = docx.Document(fpath)
        print(f'段落数: {len(doc.paragraphs)}, 表数: {len(doc.tables)}')
        if doc.tables:
            print('\n最初の表（先頭10行）:')
            for i, row in enumerate(doc.tables[0].rows):
                if i >= 10: break
                print(f'  {[c.text[:15] for c in row.cells]}')